# 00 · a constant reserve — the floor everything must clear

The simplest possible forecaster: **ignore everything and reserve the same percentage
always.** One tuned number.

It exists because it is the honest null hypothesis. Any approach that cannot beat a
single constant is not earning its complexity, and in a study with 42 days of one
person's habits that is a real possibility rather than a rhetorical one.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np, pandas as pd
import dataset as D

frame = D.build("claude")
train, validation, test = D.split(frame)
COLS = D.features(frame)
print("\ncost model:  stolen $%.2f/pct   wasted $%.2f/pct   ratio %.0f:1"
      % (D.COST_STOLEN, D.COST_WASTED, D.COST_STOLEN / D.COST_WASTED))
print("the cost-optimal quantile is therefore %.4f" % D.optimal_quantile())

12075 anchors over 41.9 days, embargo 300 min at each seam
  train        7185 rows  08-24 17:55 -> 09-18 16:35
  validation   2355 rows  09-18 21:40 -> 09-27 01:50
  test         2415 rows  09-27 06:55 -> 10-05 16:05
  dropped to the embargo: 120

cost model:  stolen $3.20/pct   wasted $0.32/pct   ratio 10:1
the cost-optimal quantile is therefore 0.9091


## The objective

Not accuracy — **money**. Two unit costs per percent of a 5-hour meter:

| | $/pct | Meaning |
|---|---|---|
| `COST_WASTED` | **0.32** | quota the bot was refused although the human never came — exactly 1% of a $32 unit |
| `COST_STOLEN` | **3.20** | quota the human wanted and the bot took — 10×, because the user has first claim |

Hyper-parameters below are chosen on **validation**, by cost. The test set is read
only in `05_comparison.ipynb`.

In [2]:
candidates = {"reserve %d%%" % c: c for c in range(0, 101, 1)}
chosen, table = D.tune("constant", candidates, validation,
                       lambda c, N: np.full(len(validation), float(c)), show=0)

N = 30   best: reserve 0%                   $0.5191   (worst $28.7481)
N = 60   best: reserve 0%                   $1.7977   (worst $28.6202)
N = 120  best: reserve 0%                   $4.8183   (worst $28.3182)
N = 180  best: reserve 13%                  $7.8913   (worst $27.9933)
N = 300  best: reserve 22%                  $13.1661   (worst $27.3254)
constant: mean cost across horizons on this frame = $5.6385 per anchor


In [3]:
# The theory says the cheapest constant is the q-quantile of demand, q = 0.909.
# This is the check.
print("  N    best constant   q90.9 of demand   <- these should agree")
for N in D.HORIZONS:
    y = validation["y%d" % N].values; y = y[~np.isnan(y)]
    print("  %3d       %3d%%             %5.1f%%" % (
        N, chosen[N][1], np.percentile(y, D.optimal_quantile() * 100)))
print()
print("Below MIN_HUMAN_RESERVE_PCT = %g the budget stage reserves the floor anyway," % D.P_FLOOR)
print("so any prediction under it costs exactly the same -- which is why the short")
print("horizons all land at the floor rather than at a positive number.")

  N    best constant   q90.9 of demand   <- these should agree
   30         0%               1.0%
   60         0%               4.0%
  120         0%               8.0%
  180        13%              13.0%
  300        22%              22.0%

Below MIN_HUMAN_RESERVE_PCT = 10 the budget stage reserves the floor anyway,
so any prediction under it costs exactly the same -- which is why the short
horizons all land at the floor rather than at a positive number.


In [4]:
val = [D.evaluate("constant", validation["y%d" % N].values,
                  np.full(len(validation), float(chosen[N][1]))) | {"N": N}
       for N in D.HORIZONS]
print(pd.DataFrame(val).set_index("N")[["cost_usd", "stolen", "wasted", "coverage"]].round(3))
D.save("00_constant", {"kind": "constant",
                       "per_horizon": {str(N): {"pct": float(chosen[N][1])}
                                       for N in D.HORIZONS}}, val)

     cost_usd  stolen  wasted  coverage
N                                      
30      0.519   0.162   0.000    89.766
60      1.798   0.562   0.000    86.157
120     4.818   1.506   0.000    81.189
180     7.891   2.198   2.677    91.507
300    13.166   3.068  10.460    91.040
froze 00_constant -> design/02_prediction/lab/results/00_constant.json


PosixPath('/Users/jeanlescut/dev/agent-quota-maximizer/design/02_prediction/lab/results/00_constant.json')